# Imports

In [17]:
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import Flatten
from keras.layers import BatchNormalization
from keras.layers import Conv2D
from keras.layers import MaxPooling2D
from keras.layers import GlobalAveragePooling2D
from keras.layers import Dropout
from keras.layers import ReLU
from keras.layers import Rescaling
from keras.layers import RandomFlip
from keras.layers import RandomRotation
from keras.layers import RandomZoom
from keras.layers import RandomColorJitter
from keras.optimizers import AdamW
import numpy as np
import pathlib
import keras

import tensorflow as tf

## Download dataset

In [18]:

DATASET_URL = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
DATASET_NAME = "flower_photos"
CACHE_DIR = "."

data_dir = pathlib.Path(
    tf.keras.utils.get_file(
        origin    = DATASET_URL,
        fname     = DATASET_NAME,
        cache_dir = CACHE_DIR,
        untar     = True,
        extract   = True,
    )
)

### Parameters

In [29]:
IMG_HEIGHT = 160
IMG_WIDTH  = 160
BATCH_SIZE = 32
SEED       = 67
MAX_INTERNAL_LAYERS = 4
ADD_SECOND_CONV_LAYER = True
ADD_ONLINE_DATA_AUGMENTATION = True

TRAIN_SPLIT = 0.75
VAL_SPLIT  = 0.15
AUTOTUNE = tf.data.AUTOTUNE

### Util functions

In [20]:
full_ds = keras.utils.image_dataset_from_directory(
  data_dir / DATASET_NAME,
  seed=SEED,
  image_size=(IMG_HEIGHT, IMG_WIDTH),
  batch_size=BATCH_SIZE)

total_batches = len(full_ds)

train_size = int(TRAIN_SPLIT * total_batches)
val_size   = int(VAL_SPLIT * total_batches)

train_ds = full_ds.take(train_size)
val_ds   = full_ds.skip(train_size).take(val_size)
test_ds  = full_ds.skip(train_size + val_size)

train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

Found 3670 files belonging to 5 classes.


### Layers

In [ ]:
ensamble = []
for i in range(5):
    model = Sequential()
    model.add(Rescaling(1./255))

    if ADD_ONLINE_DATA_AUGMENTATION:
        model.add(RandomFlip("horizontal"))
        model.add(RandomColorJitter())
        model.add(RandomRotation(0.2))
        model.add(RandomZoom(0.1))

    for j in range(5, MAX_INTERNAL_LAYERS + 5):
        model.add(Conv2D(2 ** j, 3, padding='same'))
        model.add(BatchNormalization())
        model.add(ReLU())

        if ADD_SECOND_CONV_LAYER:
            model.add(Conv2D(2 ** j, 3, padding='same'))
            model.add(BatchNormalization())
            model.add(ReLU())

        if j < MAX_INTERNAL_LAYERS + 4:
            model.add(MaxPooling2D())

    model.add(GlobalAveragePooling2D())

    model.add(Dense(2 ** (MAX_INTERNAL_LAYERS + 5 - 1), activation='relu'))
    model.add(Dropout(0.5))

    model.add(Dense(5))

    model.compile(
        optimizer=AdamW(1e-4, 1e-4),
        loss=tf.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=['accuracy'])
    
    callbacks = [
        keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=15, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_accuracy',
            factor=0.5,
            patience=5,
            min_lr=1e-6,
            verbose=1
        ),
        keras.callbacks.ModelCheckpoint(
            filepath=f"models/best_checkpoint{i}.keras",
            monitor='val_accuracy',
            mode='max',
            save_best_only=True)
    ]

    model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=100,
        callbacks=callbacks
    )

    ensamble.append(model)

Epoch 1/100


86/86 ━━━━━━━━━━━━━━━━━━━━ 97s 1s/step - accuracy: 0.5047 - loss: 1.2306 - val_accuracy: 0.2426 - val_loss: 1.9700 - learning_rate: 1.0000e-04
Epoch 2/100
86/86 ━━━━━━━━━━━━━━━━━━━━ 89s 1s/step - accuracy: 0.5919 - loss: 1.0421 - val_accuracy: 0.2426 - val_loss: 2.6189 - learning_rate: 1.0000e-04
Epoch 3/100
86/86 ━━━━━━━━━━━━━━━━━━━━ 90s 1s/step - accuracy: 0.6308 - loss: 0.9549 - val_accuracy: 0.2426 - val_loss: 2.4520 - learning_rate: 1.0000e-04
Epoch 4/100
86/86 ━━━━━━━━━━━━━━━━━━━━ 90s 1s/step - accuracy: 0.6512 - loss: 0.9114 - val_accuracy: 0.3640 - val_loss: 1.6185 - learning_rate: 1.0000e-04
Epoch 5/100
86/86 ━━━━━━━━━━━━━━━━━━━━ 90s 1s/step - accuracy: 0.6617 - loss: 0.8700 - val_accuracy: 0.4945 - val_loss: 1.1835 - learning_rate: 1.0000e-04
Epoch 6/100
86/86 ━━━━━━━━━━━━━━━━━━━━ 87s 1s/step - accuracy: 0.6831 - loss: 0.8315 - val_accuracy: 0.5294 - val_loss: 1.2034 - learning_rate: 1.0000e-04
Epoch 7/100
86/86 ━━━━━━━━━━━━━━━━━━━━ 85s 985ms/step - accuracy: 0.7053 - loss: 0

In [ ]:
def build_ensemble(models):
    inp = keras.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3))
    
    outputs = []
    for i, model in enumerate(models):
        model.name = f"model_{i}"
        outputs.append(model(inp, training=False))
    
    avg = keras.layers.Average()(outputs)
    
    return keras.Model(inputs=inp, outputs=avg)

# ensamble = [
#     keras.models.load_model("models/best_checkpoint0.keras"),
#     keras.models.load_model("models/best_checkpoint1.keras"),
#     keras.models.load_model("models/best_checkpoint2.keras"),
#     keras.models.load_model("models/best_checkpoint3.keras"),
#     keras.models.load_model("models/best_checkpoint4.keras"),
# ]

ensemble_model = build_ensemble(ensamble)

ensemble_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

for e in ensamble:
    loss, accuracy = e.evaluate(test_ds)
    print(f"Ensemble Accuracy: {accuracy:.4f}")

loss, accuracy = ensemble_model.evaluate(test_ds)
print(f"Ensemble Accuracy: {accuracy:.4f}")

In [ ]:
# tuner.search(train_ds, epochs=5, validation_data=val_ds)
# best_model = tuner.get_best_models()[0]

In [ ]:
# best_model.predict(val_ds)